# 13. PMD histone

Part of the **[Fig. 2 chapter](fig2.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import os
import glob
import cooler
import pandas as pd
import numpy as np
from multiprocessing import Pool, cpu_count
from collections import defaultdict
from scipy.stats import zscore, pearsonr, norm
from sklearn.cluster import KMeans
import anndata

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats


In [ ]:
indir = f'{ENTEX_ROOT}/'
outdir = f'{indir}analysis/PMD_histone/'
pmddir = f'{indir}analysis/PMD/tissue/'

In [ ]:
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
tissue_list = pd.Index(np.sort(['AG', 'Sk', 'EG', 'LV', 'RAA', 'B', 'ST', 'SkGcn', 'TrCo', 'LG', 'SB', 'NTb', 'PI']))


In [ ]:
meta = pd.read_csv(f'{indir}clustering/merged/5kCG100k3C_summary.csv.gz', header=0, index_col=0)
meta[['Donor', 'Tissue', 'cluster', 'subtype', 'majortype']] = meta[['Donor', 'Tissue', 'cluster', 'subtype', 'majortype']].astype(str)
group_list = (meta['Tissue'] + '+' + meta['Donor']).sort_values().unique()
group_list

### load and plot tissue PMD

In [ ]:
nc = 3
cluster_key = f'kmeans{nc}'
cluster_color = sns.color_palette('Set2', nc)
fig, axes = plt.subplots(4, 6, figsize=(6.5,8), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': np.tile([20,1], 3)})
for i,ct in enumerate(tissue_list[1:]):
    ax = axes.flatten()[i*2]
    sns.despine(ax=ax, left=True, bottom=True)
    adata = anndata.read_h5ad(f'{pmddir}{ct}_10kb_hist.h5ad')
    
    leg = np.sort(adata.obs[cluster_key].unique())
    # selc = np.random.choice(np.arange(adata.shape[0]), 1000, False)
    # idx = adata.obs[cluster_key].iloc[selc].sort_values().index
    idx = adata.obs[cluster_key].sort_values().index
    tmp = adata[idx].X
    ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
              interpolation='none', rasterized=True)
    ax.set_title(ct)
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    ax.set_xticks([-0.5, 49.5, 99.5])
    ax.set_xticklabels([0, 0.5, 1])
    ax.set_yticks([])
    
    ax = axes.flatten()[i*2+1]
    ax.axis('off')
    for k in range(nc):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')

fig.tight_layout()
# fig.savefig('mCG_distribution/L1_10kb_hist_kmeans4.pdf', transparent=True)


In [ ]:
nc = 3
res = 10000
cluster_key = f'kmeans{nc}'
pmd = {}
for ct in tissue_list:
    adata = anndata.read_h5ad(f'{pmddir}{ct}_10kb_hist.h5ad')
    pmd[ct] = adata.obs[cluster_key].copy()

pmd = pd.DataFrame(pmd)
pmd.index = pmd.index.str.split('-').str[0] + '_' + (pmd.index.str.split('-').str[2].astype(int) // res).astype(str)
pmd = pmd.loc[pmd.isna().sum(axis=1)==0]
pmd = pmd.astype(int).replace({1: 2, 2: 1})
pmd.to_hdf(f'{pmddir}tissue13_kmeans3_mccomp.hdf', key='data')


### load saved files

In [ ]:
tissue_map = {"Peyer's patch": 'SB', 
              'adrenal gland': 'AG', 
              'body of pancreas': 'PI',
              'breast epithelium': 'B', 
              'esophagus muscularis mucosa': 'EG',
              'gastrocnemius medialis': 'SkGcn', 
              'heart left ventricle': 'LV',
              'right atrium auricular region': 'RAA', 
              'stomach': 'ST', 
              'suprapubic skin': 'Sk',
              'tibial nerve': 'NTb', 
              'transverse colon': 'TrCo', 
              'upper lobe of left lung': 'LG'}
donor_map = {'ENCDO271OUW':'PT-1LVAN', 
             'ENCDO845WKR':'PT-1JKYN', 
             'ENCDO451RUA':'PT-1K2DA', 
             'ENCDO793LXB':'PT-1LGRB'}


In [ ]:
pmd = pd.read_hdf(f'{pmddir}tissue13_kmeans3_mccomp.hdf', key='data')


In [ ]:
bulk_meta = pd.read_csv('/large_storage/zhoulab/resource/ENCODE/histone/metadata.tsv', sep='\t', header=0, index_col=0)
bulk_meta = bulk_meta.loc[bulk_meta['File type']=='bigWig']
bulk_meta['donor'] = bulk_meta['Donor(s)'].str.split('/').str[2].map(donor_map)
bulk_meta['tissue'] = bulk_meta['Biosample term name'].map(tissue_map)
bulk_meta['group'] = bulk_meta['tissue'] + '+' + bulk_meta['donor']


In [ ]:
bulk_meta['group'].value_counts()


### merge histone data

In [ ]:
def load_tsv(sample):
    file_name = f'{outdir}result/{sample}.10kb.tsv'
    tmp = pd.read_csv(file_name, index_col=0, header=None, sep='\t')[5]
    tmp.name = sample
    return tmp
    

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

cpu = 24
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for sample in bulk_meta.index:
        future = executor.submit(
            load_tsv,
            sample=sample,
        )
        futures[future] = sample

    data = []
    for future in as_completed(futures):
        data.append(future.result())
        

In [ ]:
data = pd.DataFrame(data)
data = data.loc[:, data.columns.str.split('_').str[0].isin(chrom_sizes.index)].T
data.to_hdf(f'{outdir}histone_10kb.hdf', key='data')


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c7'], axis=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()


In [ ]:
epi_L1 = L1_meta.index[L1_meta['L1_abbr'].str.split(' ').str[0]=='Epi']

In [ ]:
# res = 10000
# pmd = pd.read_hdf(f'{indir}analysis/PMD/10kb/merged_kmeans4_onehot.hdf', key='data')
# pmd.columns = pmd.columns.str.split('-').str[0]
# pmd = pmd.iloc[:, np.arange(L1_meta.shape[0])*4+2] * 2 + pmd.iloc[:, np.arange(L1_meta.shape[0])*4+3]
# pmd.index = pmd.index.str.split('-').str[0] + '_' + (pmd.index.str.split('-').str[2].astype(int) // res).astype(str)
# pmd = pmd.drop(['c10','c16','c14','c31'], axis=1)
# pmd = pmd[epi_L1]

In [ ]:
bulk_meta['Biosample term name'].value_counts()

In [ ]:
bulk_meta['Experiment target'].value_counts()

### Load histone data

In [ ]:
import qnorm
data = pd.read_hdf(f'{outdir}histone_10kb.hdf', key='data')
data_all = {}
for hist in bulk_meta['Experiment target'].unique():
    sels = bulk_meta.index[(bulk_meta['Experiment target']==hist)# & (bulk_meta['group'].isin(group_list))
                           ]
    # sels = sels[sels.isin(data.columns)]
    tmp = data.loc[pmd.index, sels]
    # data_all[hist] = tmp / tmp.sum(axis=0) * 1e5
    tmp = qnorm.quantile_normalize(tmp, axis=1)
    data_all[hist] = tmp.groupby(bulk_meta.loc[sels, 'tissue'], axis=1).mean()
    print(hist, data_all[hist].shape)
    

In [ ]:
np.sort(bulk_meta['Biosample term name'].unique())

### merge all tissues/cell types

In [ ]:
data_pmd = pmd.mean(axis=1).sort_values()
data_hist = pd.DataFrame([data_all[hist].mean(axis=1) for hist in data_all.keys()], index=data_all.keys()).loc[:, data_pmd.index]
data_hist.index = data_hist.index.str.split('-').str[0]


In [ ]:
data_pmd = pd.DataFrame(data_pmd).T

In [ ]:
from scipy.stats import zscore
# sns.clustermap(result, z_score=1, row_cluster=False, vmax=2, vmin=-2, cmap='vlag', yticklabels=False)
corder = ['H3K9me3', 'H3K27me3', 'H3K36me3', 'H3K4me1', 'H3K27ac', 'H3K4me3']
fig, axes = plt.subplots(2, 1, figsize=(4,2), dpi=300, gridspec_kw={'height_ratios': [0.2, 1]})
ax = axes[0]
vmin, vmax = 0, 2
sns.heatmap(data_pmd, xticklabels=False, yticklabels=['CG Comp'], 
            cmap='vlag', ax=ax, vmin=vmin, vmax=vmax, rasterized=True,
            cbar_kws={"shrink": 1.0, "aspect": 5, "ticks": [0,1,2]})
ax = axes[1]
vmin, vmax = 0, 1
sns.heatmap(data_hist.loc[corder], xticklabels=False, yticklabels=corder, 
            cmap='vlag', ax=ax, vmin=vmin, vmax=vmax, rasterized=True,
            cbar_kws={"shrink": 0.2, "aspect": 5, "ticks": [vmin, vmax]})

for ax in axes:
    sns.despine(ax=ax, top=False, right=False, left=False, bottom=False)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    cbar = ax.collections[0].colorbar
    cbar.ax.spines['outline'].set_linewidth(0.8)
fig.tight_layout()
fig.savefig(f'PMD_histone/PMD_histone_average.pdf', transparent=True)


# sns.heatmap(result[corder].T, cmap='vlag', xticklabels=False, yticklabels=corder, vmax=2, vmin=0)


In [ ]:
from scipy.stats import zscore
# sns.clustermap(result, z_score=1, row_cluster=False, vmax=2, vmin=-2, cmap='vlag', yticklabels=False)
corder = ['H3K9me3', 'H3K27me3', 'H3K36me3', 'H3K4me1', 'H3K27ac', 'H3K4me3']
fig, axes = plt.subplots(2, 1, figsize=(4,2), dpi=300, gridspec_kw={'height_ratios': [0.2, 1]})
ax = axes[0]
vmin, vmax = 0, 2
sns.heatmap(data_pmd, xticklabels=False, yticklabels=['CG Comp'], 
            cmap='vlag', ax=ax, vmin=vmin, vmax=vmax, rasterized=True,
            cbar_kws={"shrink": 1.0, "aspect": 5, "ticks": [0,1,2]})
ax = axes[1]
vmin, vmax = 0, 1
sns.heatmap(data_hist.loc[corder], xticklabels=False, yticklabels=corder, 
            cmap='vlag', ax=ax, vmin=vmin, vmax=vmax, rasterized=True,
            cbar_kws={"shrink": 0.2, "aspect": 5, "ticks": [vmin, vmax]})

for ax in axes:
    sns.despine(ax=ax, top=False, right=False, left=False, bottom=False)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    cbar = ax.collections[0].colorbar
    cbar.ax.spines['outline'].set_linewidth(0.8)
fig.tight_layout()
fig.savefig(f'PMD_histone/PMDtissue_histone_average.pdf', transparent=True)


# sns.heatmap(result[corder].T, cmap='vlag', xticklabels=False, yticklabels=corder, vmax=2, vmin=0)


### plot bin-by-tissue heatmap

In [ ]:
def order_row(data, nc):
    data = data.reset_index(drop=True)
    
    # Perform KMeans clustering    
    kmeans = KMeans(n_clusters=nc, random_state=0, n_init=10)
    clusters = kmeans.fit_predict(data)

    # Create a new dataframe with cluster labels
    cluster_df = pd.DataFrame({'Cluster': clusters}, index=data.index)

    # Sort the data by cluster and plot the heatmap
    merged_data = pd.concat([data, cluster_df], axis=1).groupby(by='Cluster').mean()
    cg = sns.clustermap(merged_data, cmap='vlag', figsize=(6,4))
    
    rorder = cg.dendrogram_row.reordered_ind.copy()
    corder = cg.dendrogram_col.reordered_ind.copy()
    
    leg = merged_data.index[rorder]
    count = pd.Series(clusters).value_counts().loc[leg]
    sorted_data = pd.concat([data[clusters==i] for i in leg], axis=0)
    
    return sorted_data, count, cluster_df, corder


In [ ]:
nc = 8
group_palette = sns.color_palette('Set3', nc)
tmp, count, labelpmd, corder_pmd = order_row(pmd, nc=nc)



In [ ]:
corder_pmd = pmd.columns[corder_pmd]
ncol = len(data_all)+1
fig, axes = plt.subplots(1, ncol, sharey='all', figsize=(3*ncol,3), dpi=300)

ax = axes[0]
sns.heatmap(tmp.loc[:, corder_pmd], cmap='vlag', vmin=0, vmax=2, ax=ax, 
            xticklabels=corder_pmd, yticklabels='', rasterized=True)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_title('PMD', fontsize=10)

for i,hist in enumerate(data_all):
    ax = axes[i+1]
    data = data_all[hist]
    corder_his = corder_pmd[corder_pmd.isin(data.columns)]
    sns.heatmap(data.iloc[tmp.index][corder_his], cmap='vlag', vmin=0, vmax=1, ax=ax, 
                xticklabels=corder_his, yticklabels='', rasterized=True)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_title(hist.split('-')[0], fontsize=10)
    
fig.tight_layout()


In [ ]:
selb = np.random.choice(pmd.index, 5000, False)
# selb = pmd.index[labelpmd.groupby('Cluster').sample(n=500, random_state=0).index]
cg = sns.clustermap(pmd.loc[selb], figsize=(0.1, 0.1))
corder_pmd = cg.dendrogram_col.reordered_ind.copy()
rorder = cg.dendrogram_row.reordered_ind.copy()
corder = {}
for hist in data_all.keys():
    data = data_all[hist]
    cg = sns.clustermap(data.loc[selb], metric='cosine', figsize=(0.1, 0.1))
    corder[hist] = cg.dendrogram_col.reordered_ind.copy()


In [ ]:
ncol = len(corder)+1
fig, axes = plt.subplots(1, ncol, sharey='all', figsize=(2.5*ncol,3), dpi=300)

ax = axes[0]
sns.heatmap(pmd.loc[selb[rorder]].iloc[:, corder_pmd], cmap='coolwarm', vmin=0, vmax=2, ax=ax, 
            xticklabels=pmd.columns[corder_pmd], yticklabels='', rasterized=True)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_title('PMD', fontsize=10)

for i,hist in enumerate(corder):
    ax = axes[i+1]
    data = data_all[hist]
    corder_his = corder[hist]
    sns.heatmap(data.loc[selb[rorder]].iloc[:, corder_his], cmap='vlag', vmin=0, vmax=1, ax=ax, 
                xticklabels=data.columns[corder_his], yticklabels='', rasterized=True)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_title(hist.split('-')[0], fontsize=10)
    
fig.tight_layout()


In [ ]:
ncol = len(data_all)+1
fig, axes = plt.subplots(1, ncol, sharey='all', figsize=(2*ncol,3), dpi=300)

ax = axes[0]
sns.heatmap(pmd.loc[selb[rorder]].iloc[:, corder_pmd], cmap='vlag', vmin=0, vmax=2, ax=ax, 
            xticklabels=pmd.columns[corder_pmd], yticklabels='', rasterized=True, cbar=False)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_title('PMD', fontsize=10)

for i,hist in enumerate(['H3K9me3', 'H3K27me3', 'H3K36me3', 'H3K4me1', 'H3K27ac', 'H3K4me3']):
    ax = axes[i+1]
    data = data_all[f'{hist}-human']
    corder_his = pmd.columns[corder_pmd]
    corder_his = corder_his[corder_his.isin(data.columns)]
    sns.heatmap(data.loc[selb[rorder], corder_his], cmap='vlag', vmin=0, vmax=1, ax=ax, 
                xticklabels=corder_his, yticklabels='', rasterized=True, cbar=False)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_title(hist, fontsize=10)
    
for ax in axes:
    sns.despine(ax=ax, top=False, right=False, left=False, bottom=False)

fig.tight_layout()
fig.savefig(f'PMD_histone/PMD_histone_tissue.svg', transparent=True)


In [ ]:
pmd.columns[corder_pmd].map(L1_meta['L1_annot'])


In [ ]:
print(pmd.shape, labelpmd.shape)

In [ ]:
selb = np.random.choice(pmd.index, 5000, False)
# selb = pmd.index[labelpmd.groupby('Cluster').sample(n=500, random_state=0).index]
cg = sns.clustermap(pmd.loc[selb], figsize=(0.1, 0.1))
corder_pmd = cg.dendrogram_col.reordered_ind.copy()
rorder = cg.dendrogram_row.reordered_ind.copy()
corder = {}
for hist in data_all.keys():
    data = data_all[hist]
    cg = sns.clustermap(data.loc[selb], metric='cosine', figsize=(0.1, 0.1))
    corder[hist] = cg.dendrogram_col.reordered_ind.copy()


In [ ]:
ncol = len(corder)+1
fig, axes = plt.subplots(1, ncol, sharey='all', figsize=(5*ncol,5), dpi=300)

ax = axes[0]
sns.heatmap(pmd.loc[selb[rorder]].iloc[:, corder_pmd], cmap='coolwarm', vmin=0, vmax=2, ax=ax, 
            xticklabels=pmd.columns[corder_pmd], yticklabels='')
ax.set_xticklabels(ax.get_xticklabels(), rotation=270)
ax.set_title('PMD', fontsize=10)

for i,hist in enumerate(corder):
    ax = axes[i+1]
    data = data_all[hist]
    corder_his = corder[hist]
    sns.heatmap(data.loc[selb[rorder]].iloc[:, corder_his], cmap='vlag', vmin=0, vmax=1, ax=ax, 
                xticklabels=data.columns.map(bulk_meta['tissue'])[corder_his], yticklabels='')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=270)
    ax.set_title(hist.split('-')[0], fontsize=10)
    
fig.tight_layout()


In [ ]:
ncol = len(corder)+1
fig, axes = plt.subplots(1, ncol, sharey='all', figsize=(5*ncol,5), dpi=300)

ax = axes[0]
sns.heatmap(pmd.loc[selb[rorder]].iloc[:, corder_pmd], cmap='coolwarm', vmin=0, vmax=2, ax=ax, 
            xticklabels=pmd.columns.map(L1_meta['L1_annot'])[corder_pmd], yticklabels='')
ax.set_xticklabels(ax.get_xticklabels(), rotation=270)
ax.set_title('PMD', fontsize=10)

for i,hist in enumerate(corder):
    ax = axes[i+1]
    data = data_all[hist]
    corder_his = corder[hist]
    sns.heatmap(data.loc[selb[rorder]].iloc[:, corder_his], cmap='vlag', vmin=0, vmax=1, ax=ax, 
                xticklabels=data.columns.map(bulk_meta['Biosample term name'])[corder_his], yticklabels='')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=270)
    ax.set_title(hist.split('-')[0], fontsize=10)
    
fig.tight_layout()


In [ ]:
from sklearn.metrics import pairwise_distances
pairwise_distances(pmd.T, data.T, metric='correlation').mean()

In [ ]:
# selb = np.random.choice(pmd.index, 5000, False)
data = data_all['H3K9me3-human']
selb = data.std(axis=1).sort_values(ascending=False).index[:5000]
data = data.loc[selb]
print(data.shape)

In [ ]:
sns.clustermap(data, metric='correlation', cmap='vlag', vmax=1, vmin=0,
               xticklabels=data.columns.map(bulk_meta['Biosample term name']))


In [ ]:
hist = 'H3K27ac-human'
data = pd.read_hdf(f'{outdir}histone_10kb.hdf', key='data')
selc = bulk_meta.index[bulk_meta['Experiment target']==hist]
datatmp = data[selc]
designtmp = bulk_meta.loc[selc].rename(columns={'Biosample term name': 'tissue', 'Donor(s)': 'donor'})
subclass_list = bulk_meta.loc[selc, 'Biosample term name'].unique()
subclass_list

In [ ]:
import itertools

for ct1, ct2 in itertools.combinations(subclass_list, 2):
    group_name = f'{ct1}+{ct2}'.replace(' ', '_').replace('/', '-')
    if os.path.exists(f'{outdir}diff_10k/{hist}/{group_name}.stats.hdf'):
        stat_res = pd.read_hdf(f'{outdir}diff_10k/{hist}/{group_name}.stats.hdf', key='data')
        if (stat_res['padj']<0.01).sum()>100:
            print(f'"{ct1}", "{ct2}"', (stat_res['padj']<0.01).sum())
    

In [ ]:
ct1, ct2 = "upper lobe of left lung", "breast epithelium"
stat_res = pd.read_hdf(f'{outdir}diff_10k/{hist}/{group_name}.stats.hdf', key='data')
selb = stat_res.index[stat_res['padj']<0.01]
selb = np.random.choice(selb, 5000, False)
tmp = datatmp.loc[selb, designtmp['tissue'].isin([ct1, ct2])]


In [ ]:
cg = sns.clustermap(tmp, metric='correlation', cmap='vlag', vmax=2, vmin=-2, z_score=0
               xticklabels=tmp.columns.map(bulk_meta['Biosample term name']))
rorder = tmp.index[cg.dendrogram_row.reordered_ind]


In [ ]:
res = 10000
adata1 = anndata.read_h5ad(f'{pmddir}LG_10kb_hist.h5ad')
adata1.obs.index = adata1.obs.index.str.split('-').str[0] + '_' + (adata1.obs.index.str.split('-').str[2].astype(int) // res).astype(str)

adata2 = anndata.read_h5ad(f'{pmddir}B_10kb_hist.h5ad')
adata2.obs.index = adata2.obs.index.str.split('-').str[0] + '_' + (adata2.obs.index.str.split('-').str[2].astype(int) // res).astype(str)

rorder = rorder[rorder.isin(adata1.obs.index) & rorder.isin(adata2.obs.index)]
print(len(rorder))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6,4), dpi=300, sharex='all', sharey='all')
for i,adata in enumerate([adata1, adata2]):
    ax = axes[i]
    tmp = adata[rorder].X
    sns.heatmap(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), ax=ax)


In [ ]:
rorder = rorder[rorder.isin(pmd.index)]
sns.heatmap(pmd.loc[rorder, ['B', 'LG']], vmin=0, vmax=2, cmap='vlag')


In [ ]:
fig, ax = plt.subplots(figsize=(3,1.5), dpi=300)
sns.histplot(data.sum(axis=0).sort_values(), bins=100, ax=ax)

In [ ]:
for hist,df in bulk_meta.groupby('Experiment target'):
    print(hist, len(df['Biosample term name'].unique()))


In [ ]:
bulk_meta['Biosample term name'].value_counts()